# Testing of MoNF model on synthetic point clouds

## Notes

* Initially a copy from github://sfikas/normalizing-flows/05-MoNF_refactored.ipynb
  * (Probabilistic Relative Pose - Mixture of Normalizing Flows, test n1)


In [1]:
import torch
import numpy as np
import normflows as nf
from matplotlib import pyplot as plt
from tqdm import tqdm
np.set_printoptions(formatter={'float': lambda x: "{0:0.03f}".format(x)})
from bayesrpe import count_parameters, createSyntheticDataset
from bayesrpe import EngelsNister
from bayesrpe import createFreshFlows, trainFlow, plot_flows

def computeMinAngularError(est, gt):
    '''
    est:    Vector of estimates, expressed in radians.
    gt:     Vector of ground truth, expressed in radians.

    result: Computes angular error between all pairs of (x,y), where x \in est, y \in gt.
    Returns minimum value.

    Το κακό με αυτό το metric είναι ότι θα δώσει καλή τιμή αρκεί να έχει ματσαριστεί καλά *μία* τουλάχιστον composing motion.
    (αν οι υπόλοιπες είναι χάλια, αυτό δεν φαίνεται)
    '''
    assert(type(est) is np.ndarray) 
    assert(type(gt) is np.ndarray)
    argmin_est = None
    argmin_gt = None
    min_error = +np.inf
    for x in est:
        for y in gt:
            err = np.abs(np.rad2deg(x)-np.rad2deg(y))
            if(err < min_error):
                min_error = err
                argmin_est = x
                argmin_gt = y
    return(min_error, argmin_est, argmin_gt)

def computeMeanAngularError(est, gt, evaluate_over_all_gt_angles=True):
    '''
    est:    Vector of estimates, expressed in radians.
    gt:     Vector of ground truth, expressed in radians.

    result: Computes angular error between all pairs of (x,y), where x \in est, y \in gt.
    Returns mean value.
    '''
    assert(type(est) is np.ndarray) 
    assert(type(gt) is np.ndarray)
    argmin_est = None
    argmin_gt = None
    errors = []
    if not evaluate_over_all_gt_angles:
        gt,est = est,gt
    for y in gt:    
        min_error = +np.inf        
        for x in est:
            err = np.abs(np.rad2deg(x)-np.rad2deg(y))
            if(err < min_error):
                min_error = err
                #argmin_est = x
                #argmin_gt = y
        errors.append(min_error)
    return(np.mean(errors))


In [2]:
def emAlgorithm(
        x2_A, x2_B, num_kernels, em_iterations = 5, use_tqdm = False
):
    enable_cuda = True
    device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')
    print(f'*** Using device: {device}. ***')
    L = 10**3              # Number of samplings of E, using the learned flows. This is used to update "z" and "sigma".
    total_points = x2_A.shape[-1]
    with torch.no_grad():
        z = torch.zeros([num_kernels, total_points])
        u = torch.ones([num_kernels, total_points])
        z[0, :] = torch.rand(total_points,)
        #tt = (z[0, :] > 0.5)
        #z[0, tt] = 1.
        z[1, :] = 1 - z[0, :]
        w = torch.ones(num_kernels,) / num_kernels
        sigma2 = 1e-2 * torch.ones(num_kernels,)

    z = z.to(device)
    w = w.to(device)
    u = u.to(device)
    sigma2 = sigma2.to(device)

    if device == torch.device('cuda'):
        samples_per_flow_iteration = 2**14
        max_flow_iterations = 5
    else:
        samples_per_flow_iteration = 2**8
        max_flow_iterations = 100
        
    flowmodels = [None]*num_kernels
    flow_layers = createFreshFlows(num_kernels=num_kernels)
    # We use a Uniform distribution as the base [Rezende 2020], with non-negative density in [-1, 1].
    #base = nf.distributions.Uniform(zmin=-np.pi, zmax=+np.pi) #TODO this is buggy actually? raise issue?
    #base = nf.distributions.UniformGaussian(1, [0], torch.tensor([np.pi]))
    #likelihood_formula = 'generalized-gaussian'
    likelihood_formula = 'engels-nister'
    likelihood_formula_nosigma = 'generalized-gaussian-nosigma'
    from IPython import display
    for iter in range(em_iterations):
        #############################################################
        # 
        # E-step
        #
        # The E-step comprises of optimizing q(E) and q(z).
        # (note: I deliberately optimize q(E) first, so that an initial flow exists, in order to use for sampling on the step that optimizes q(z))
        #
        # q(E)
        #
        # In a nutshell, here we do standard VI with Normalizing flows (Rezende-Mohammed 15),
        # *but* the samples will need to be weighted by z[:,:] (*and* u[:,:], if we assume the robust model.)
        #
        # eg on GMMs this is a non-issue, because the kernel parameters are updated on each round via a closed-form, globally optimal formula.
        for k in range(num_kernels):
            flowmodels[k] = trainFlow(
                                    x2_A, x2_B, 
                                    z[k, :] * u[k, :],
                                    sigma2[k],
                                    E_prior = None,
                                    likelihood_formula=likelihood_formula,
                                    base = nf.distributions.UniformGaussian(1, [0], torch.tensor([np.pi])),
                                    #target = EngelsNister(x2_A=x2_A, x2_B=x2_B, z=z[k, :] * u[k, :], sigma2=sigma2[k],
                                    #                      likelihood_formula='generalized-gaussian'),
                                    flow_layers=flow_layers[k], #=None will re-initialize flow parameters on each iteration (have to check if works as intended..)
                                    #initialization=flowmodels[k],
                                    initialization=None,
                                    num_samples=samples_per_flow_iteration,
                                    max_iter = max_flow_iterations,
                                    enable_cuda=enable_cuda,
                                    use_tqdm=use_tqdm,
                                    learning_rate=5e-3)
            flowmodels[k].eval()
        #plot_flows(flowmodels, wait=display)


        #
        # q(z)
        #
        # Each z object is shaped as num_kernels x N
        # First initialize as logπ (shaped as 1xnum_kernels).
        # (this is the first summation term)
        with torch.no_grad():
            logz = torch.outer(torch.log(w), torch.ones(total_points).to(device))
        # The next summation term depends on our likelihood definition;
        # we must omit "z" terms (and not forget to include "u"!)
        for k in range(num_kernels):
            unweighted_likelihood = EngelsNister(x2_A, x2_B, z = u[k,:], k=1, sigma2=sigma2[k], 
                                                    likelihood_formula=likelihood_formula, 
                                                    sum_over_x=False).to(device)
            # Note: Use my ".log_prob(batch)" option here if necessary (λογικά σε 5DOF Essential θα χρειαστεί)
            # (or, just sample batches using Flow.sample(.) repeatedly)
            sampled_E, _ = flowmodels[k].sample(num_samples = L)
            logz[k, :] += torch.mean(unweighted_likelihood.log_prob(sampled_E), dim=1)
        
        ######## ENDOF -- Now normalize (uses my trick from Bayesian GMM)
        with torch.no_grad():
            pseudoZ = torch.zeros_like(logz)
            for j in range(num_kernels):
                for k in range(num_kernels):
                    pseudoZ[k, :] = logz[k, :] - logz[j, :]
                #######
                z[j, :] = 1.0 / torch.sum(torch.exp(pseudoZ), dim=0)
            ######### ENDOF

        #
        # q(U)
        #
        # Each U object is shaped as num_kernels x N, like z.
        with torch.no_grad():
            vj = torch.Tensor([1]).to(device)
            for k in range(num_kernels):
                unweighted_likelihood = EngelsNister(x2_A, x2_B, z = z[k, :], k=1, sigma2=sigma2[k],    # was: z=None (though mistaken in theory)
                                                        likelihood_formula=likelihood_formula, 
                                                        sum_over_x=False).to(device)
                sampled_E, _ = flowmodels[k].sample(num_samples = L)
                u[k, :] = vj + torch.mean(unweighted_likelihood.log_prob(sampled_E), dim=1)
                u[k, :] = (vj + 3) / u[k, :]


        # M-step
        # The M-step comprises of optimizing sigma and w (π in the paper)
        # Update for π
        if False:
            for k in range(num_kernels):
                w[k] = torch.mean(z[k,:])
            # ("Gimmick": Add a minimum value)
            w += 1e-2
            w = w / torch.sum(w)

        # Update for sigma
        #
        with torch.no_grad():
            for k in range(num_kernels):
                unweighted_likelihood = EngelsNister(x2_A, x2_B, z = z[k,:]*u[k,:], k=1, sigma2=sigma2[k], 
                                                    likelihood_formula=likelihood_formula_nosigma, sum_over_x=False
                                                    ).to(device)
                sampled_E, _ = flowmodels[k].sample(num_samples = L)
                tt = unweighted_likelihood.log_prob(sampled_E)
                #TODO check numerator "2"
                #sigma2[k] = torch.mean(tt) + 1e-4 # Adds this slight value to avoid NaN!
                sigma2[k] = 2 * torch.mean(tt) + 1e-4 # Adds this slight value to avoid NaN!

        #print(f'Iteration {iter}:')
        estimates = []
        for idx, model in enumerate(flowmodels):
            model.eval()
            tt_z, tt_probz = model.sample(num_samples=10000)
            tt = tt_z[torch.argmax(tt_probz)]
            estimates.append(tt.cpu().detach().numpy()[0])
            estimated_prob_XYangle = torch.rad2deg(tt.cpu()).detach().numpy()[0]
            #print(f'Kernel {idx}: Estimated angle over XY plane: {estimated_prob_XYangle} degrees ({np.deg2rad(estimated_prob_XYangle)} radians).')
            #print(f'π = {w}, σ^2 = {sigma2}')
            model.train()
    ########### ENDOF em iterations
    return(estimates)

In [3]:
for num_kernels in range(2,7):
    #num_kernels = 2
    N_points = int(1e2) #Number of points for each group that moves along its own CM (so K*CM in total)
    baseline_angle = np.array([-np.pi/3, +np.pi/4, -np.pi/6, +3*np.pi/7, -np.pi/8, +3*np.pi/8])
    baseline_angle = baseline_angle[:num_kernels]
    import cv2 as cv
    USE_CVFINDESSENTIAL = 10
    USE_BAYESIAN = 100
    standard_methods = [
        #cv.FM_LMEDS,
        #cv.FM_7POINT,
        cv.FM_8POINT,
        #cv.FM_RANSAC,
        cv.RANSAC + USE_CVFINDESSENTIAL,
        cv.LMEDS + USE_CVFINDESSENTIAL,
        USE_BAYESIAN
    ]
    print(f'Ground truth is: {baseline_angle}')
    runs = 30
    for method in standard_methods:
        min_error_list = []
        mean_error_list = []
        mean_error_list2 = []
        best_est_list = [] 
        best_gt_list = []
        for _ in tqdm(range(runs)):
            X3, x2_A, x2_B = createSyntheticDataset(baseline_length = [10]*num_kernels, baseline_angle = baseline_angle, 
                                            N_points=N_points, spatialPointPlaneDistance=20, spatialPointVariance=8)
            all_are_inliers = False
            if method == cv.FM_LMEDS:
                method_name = 'Least Median-of-squares'
            elif method == cv.FM_7POINT:
                method_name = '7-point (wout/ RANSAC) '
            elif method == cv.FM_8POINT:
                method_name = '8-point (wout/ RANSAC) '
            elif method == cv.FM_RANSAC:
                method_name = '7-point (w/ RANSAC)    '
            elif method == cv.RANSAC + 10:
                method_name = '5-point (w/ RANSAC)    '
            elif method == cv.LMEDS + 10:
                method_name = '5-point (Least MedS)   '
            elif method == USE_BAYESIAN:
                method_name = 'Ours'
            else:
                raise ValueError
            
            if method != USE_BAYESIAN:
                cur_x2A = x2_A[0:2, :].T
                cur_x2B = x2_B[0:2, :].T
                estimated_angle = []
                while not all_are_inliers:
                    if(cur_x2A.shape[0] < 8):
                        # Too few points anyway..
                        break
                    if method == cv.FM_LMEDS or method == cv.FM_RANSAC or method == cv.FM_7POINT or method == cv.FM_8POINT:
                        E, mask = cv.findFundamentalMat(cur_x2A, cur_x2B, method)
                    else:
                        E, mask = cv.findEssentialMat(cur_x2A, cur_x2B, cameraMatrix=np.eye(3), method=method - USE_CVFINDESSENTIAL)                        
                    # Actually also -t is a solution [cf Hartley&Zisserman]
                    # This corresponds to the same angle, +- π radians. (*not* *minus* radians!)
                    R1, R2, t = cv.decomposeEssentialMat(E)
                    estimated_angle.append(np.squeeze(np.arctan(t[1]/t[0]))) #TODO Maybe use atan2?
                    if(np.count_nonzero(mask) == len(mask)):
                        # mask: Κρατάει ως true τα inliers.
                        all_are_inliers = True
                        break
                    else:
                        # Σε αυτή την περίπτωση υπάρχουν outliers. Οπότε διώχνουμε τα inliers και
                        # επαναλαμβάνουμε την εκτίμηση επί των outliers.
                        #print(f'Counted {np.count_nonzero(mask)} inliers out of {len(mask)} total points.')
                        outliers = 1-np.squeeze(mask)
                        cur_x2A = np.squeeze(cur_x2A[np.where(outliers), :])
                        cur_x2B = np.squeeze(cur_x2B[np.where(outliers), :])
                        #print(f'Continuing with {cur_x2A.shape[0]} points.')                    
            else:
                print(x2_A.shape, x2_B.shape)
                estimated_angle = emAlgorithm(x2_A, x2_B, num_kernels, em_iterations = 20)
            ##################################################################################
            estimated_angle = np.array(estimated_angle)            
            ################################### ENDOF while not all_are_inliers
            min_error, best_est, best_gt = computeMinAngularError(estimated_angle, baseline_angle)
            mean_error = computeMeanAngularError(estimated_angle, baseline_angle)
            mean_error2 = computeMeanAngularError(estimated_angle, baseline_angle, evaluate_over_all_gt_angles=False)
            min_error_list.append(min_error)
            mean_error_list.append(mean_error)
            mean_error_list2.append(mean_error2)
            best_est_list.append(best_est)
            best_gt_list.append(best_gt)

        #print(f'**Method {method_name} ** / For K={num_kernels} / Minimum angular error={min_error:0.02f}° (est={best_est:0.02f} rads, gt={best_gt:0.02f} rads)')
        print(f'**Method {method_name} ** / For K={num_kernels} / estimate (on last run, indicative:) = {estimated_angle}')
        print(f'**Method {method_name} ** / For K={num_kernels} / Minimum angular error={np.mean(min_error_list):0.02f}° +- {np.std(min_error_list):0.02f}° (mean over {runs} runs)') #(est={best_est:0.02f} rads, gt={best_gt:0.02f} rads)')  
        print(f'**Method {method_name} ** / For K={num_kernels} / Mean angular error={np.mean(mean_error_list):0.02f}° +- {np.std(mean_error_list):0.02f}° (mean over {runs} runs)') #(est={best_est:0.02f} rads, gt={best_gt:0.02f} rads)')  
        print(f'**Method {method_name} ** / For K={num_kernels} / Mean angular error={np.mean(mean_error_list2):0.02f}° +- {np.std(mean_error_list2):0.02f}° (mean over {runs} runs)') #(est={best_est:0.02f} rads, gt={best_gt:0.02f} rads)')
        #if all_are_inliers:
        #    print(f'**Method {method_name} ** : All matches were deemed to be inliers')
        print('='*120)


Ground truth is: [-1.047 0.785]


100%|██████████| 30/30 [00:00<00:00, 2035.97it/s]


**Method 8-point (wout/ RANSAC)  ** / For K=2 / estimate (on last run, indicative:) = [-1.550]
**Method 8-point (wout/ RANSAC)  ** / For K=2 / Minimum angular error=28.22° +- 8.78° (mean over 30 runs)
**Method 8-point (wout/ RANSAC)  ** / For K=2 / Mean angular error=80.72° +- 8.78° (mean over 30 runs)
**Method 8-point (wout/ RANSAC)  ** / For K=2 / Mean angular error=28.22° +- 8.78° (mean over 30 runs)


100%|██████████| 30/30 [00:00<00:00, 742.09it/s]


**Method 5-point (w/ RANSAC)     ** / For K=2 / estimate (on last run, indicative:) = [-1.484]
**Method 5-point (w/ RANSAC)     ** / For K=2 / Minimum angular error=20.45° +- 13.26° (mean over 30 runs)
**Method 5-point (w/ RANSAC)     ** / For K=2 / Mean angular error=65.86° +- 13.99° (mean over 30 runs)
**Method 5-point (w/ RANSAC)     ** / For K=2 / Mean angular error=20.45° +- 13.26° (mean over 30 runs)


...

(3, 200) (3, 200)
*** Using device: cpu. ***


  3%|▎         | 1/30 [01:50<53:27, 110.59s/it]

(3, 200) (3, 200)
*** Using device: cpu. ***


  3%|▎         | 1/30 [03:14<1:33:46, 194.02s/it]


KeyboardInterrupt: 

In [ ]:
####
#
# EM algorithm
#
#
#       num_kernels             This is the number of kernels, aka "K" in the paper.
#                               I suppose that this is set to the gt number of CMs.
#       z                       Responsibilities, aka "W" in the paper.
#                               This is the expectation of values that tell us which point pairs correspond to which CM kernels.
#                               This is shaped as num_kernels x total_points (where total_points = N_points*K)
#       w                       Mixing weights of Normalizing flows, aka "π" in the paper. Shaped as 1xK.
#
#####
enable_cuda = True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')
print(f'*** Using device: {device}. ***')
L = 10**3              # Number of samplings of E, using the learned flows. This is used to update "z" and "sigma".
total_points = x2_A.shape[-1]
with torch.no_grad():
    z = torch.zeros([num_kernels, total_points])
    u = torch.ones([num_kernels, total_points])
    z[0, :] = torch.rand(total_points,)
    #tt = (z[0, :] > 0.5)
    #z[0, tt] = 1.
    z[1, :] = 1 - z[0, :]
    w = torch.ones(num_kernels,) / num_kernels
    sigma2 = 1e-2 * torch.ones(num_kernels,)

z = z.to(device)
w = w.to(device)
u = u.to(device)
sigma2 = sigma2.to(device)

if device == torch.device('cuda'):
    samples_per_flow_iteration = 2**14
    max_flow_iterations = 5
    use_tqdm = True
else:
    samples_per_flow_iteration = 2**8
    max_flow_iterations = 100
    use_tqdm = True
    
flowmodels = [None]*num_kernels
flow_layers = createFreshFlows(num_kernels=num_kernels)
# We use a Uniform distribution as the base [Rezende 2020], with non-negative density in [-1, 1].
#base = nf.distributions.Uniform(zmin=-np.pi, zmax=+np.pi) #TODO this is buggy actually? raise issue?
#base = nf.distributions.UniformGaussian(1, [0], torch.tensor([np.pi]))

em_iterations = 20
#likelihood_formula = 'generalized-gaussian'
likelihood_formula = 'engels-nister'
likelihood_formula_nosigma = 'generalized-gaussian-nosigma'
from IPython import display
for iter in range(em_iterations):
    #############################################################
    # 
    # E-step
    #
    # The E-step comprises of optimizing q(E) and q(z).
    # (note: I deliberately optimize q(E) first, so that an initial flow exists, in order to use for sampling on the step that optimizes q(z))
    #
    # q(E)
    #
    # In a nutshell, here we do standard VI with Normalizing flows (Rezende-Mohammed 15),
    # *but* the samples will need to be weighted by z[:,:] (*and* u[:,:], if we assume the robust model.)
    #
    # eg on GMMs this is a non-issue, because the kernel parameters are updated on each round via a closed-form, globally optimal formula.
    for k in range(num_kernels):
        flowmodels[k] = trainFlow(
                                  x2_A, x2_B, 
                                  z[k, :] * u[k, :],
                                  sigma2[k],
                                  E_prior = None,
                                  likelihood_formula=likelihood_formula,
                                  base = nf.distributions.UniformGaussian(1, [0], torch.tensor([np.pi])),
                                  #target = EngelsNister(x2_A=x2_A, x2_B=x2_B, z=z[k, :] * u[k, :], sigma2=sigma2[k],
                                  #                      likelihood_formula='generalized-gaussian'),
                                  flow_layers=flow_layers[k], #=None will re-initialize flow parameters on each iteration (have to check if works as intended..)
                                  #initialization=flowmodels[k],
                                  initialization=None,
                                  num_samples=samples_per_flow_iteration,
                                  max_iter = max_flow_iterations,
                                  enable_cuda=enable_cuda,
                                  use_tqdm=use_tqdm,
                                  learning_rate=5e-3)
        flowmodels[k].eval()
    plot_flows(flowmodels, wait=display)


    #
    # q(z)
    #
    # Each z object is shaped as num_kernels x N
    # First initialize as logπ (shaped as 1xnum_kernels).
    # (this is the first summation term)
    with torch.no_grad():
        logz = torch.outer(torch.log(w), torch.ones(total_points).to(device))
    # The next summation term depends on our likelihood definition;
    # we must omit "z" terms (and not forget to include "u"!)
    for k in range(num_kernels):
        unweighted_likelihood = EngelsNister(x2_A, x2_B, z = u[k,:], k=1, sigma2=sigma2[k], 
                                                likelihood_formula=likelihood_formula, 
                                                sum_over_x=False).to(device)
        # Note: Use my ".log_prob(batch)" option here if necessary (λογικά σε 5DOF Essential θα χρειαστεί)
        # (or, just sample batches using Flow.sample(.) repeatedly)
        sampled_E, _ = flowmodels[k].sample(num_samples = L)
        logz[k, :] += torch.mean(unweighted_likelihood.log_prob(sampled_E), dim=1)
    
    ######## ENDOF -- Now normalize (uses my trick from Bayesian GMM)
    with torch.no_grad():
        pseudoZ = torch.zeros_like(logz)
        for j in range(num_kernels):
            for k in range(num_kernels):
                pseudoZ[k, :] = logz[k, :] - logz[j, :]
            #######
            z[j, :] = 1.0 / torch.sum(torch.exp(pseudoZ), dim=0)
        ######### ENDOF

    #
    # q(U)
    #
    # Each U object is shaped as num_kernels x N, like z.
    with torch.no_grad():
        vj = torch.Tensor([1]).to(device)
        for k in range(num_kernels):
            unweighted_likelihood = EngelsNister(x2_A, x2_B, z = z[k, :], k=1, sigma2=sigma2[k],    # was: z=None (though mistaken in theory)
                                                    likelihood_formula=likelihood_formula, 
                                                    sum_over_x=False).to(device)
            sampled_E, _ = flowmodels[k].sample(num_samples = L)
            u[k, :] = vj + torch.mean(unweighted_likelihood.log_prob(sampled_E), dim=1)
            u[k, :] = (vj + 3) / u[k, :]


    # M-step
    # The M-step comprises of optimizing sigma and w (π in the paper)
    # Update for π
    if False:
        for k in range(num_kernels):
            w[k] = torch.mean(z[k,:])
        # ("Gimmick": Add a minimum value)
        w += 1e-2
        w = w / torch.sum(w)

    # Update for sigma
    #
    with torch.no_grad():
        for k in range(num_kernels):
            unweighted_likelihood = EngelsNister(x2_A, x2_B, z = z[k,:]*u[k,:], k=1, sigma2=sigma2[k], 
                                                likelihood_formula=likelihood_formula_nosigma, sum_over_x=False
                                                ).to(device)
            sampled_E, _ = flowmodels[k].sample(num_samples = L)
            tt = unweighted_likelihood.log_prob(sampled_E)
            #TODO check numerator "2"
            #sigma2[k] = torch.mean(tt) + 1e-4 # Adds this slight value to avoid NaN!
            sigma2[k] = 2 * torch.mean(tt) + 1e-4 # Adds this slight value to avoid NaN!

    print(f'Iteration {iter}:')
    estimates = []
    for idx, model in enumerate(flowmodels):
        model.eval()
        tt_z, tt_probz = model.sample(num_samples=10000)
        tt = tt_z[torch.argmax(tt_probz)]
        estimates.append(tt.cpu().detach().numpy()[0])
        estimated_prob_XYangle = torch.rad2deg(tt.cpu()).detach().numpy()[0]
        print(f'Kernel {idx}: Estimated angle over XY plane: {estimated_prob_XYangle} degrees ({np.deg2rad(estimated_prob_XYangle)} radians).')
        print(f'π = {w}, σ^2 = {sigma2}')
        model.train()

In [ ]:
#print(torch.count_nonzero(u < 3.9))
for k in range(num_kernels):
    plt.plot(torch.squeeze(u[k,:]).cpu().detach().numpy())
print(num_kernels)
print(np.sort(estimates))
print(np.sort(baseline_angle))
min_error, best_est, best_gt = computeMinAngularError(np.array(estimates), baseline_angle)
print(f'**Method Bayes-RPE ** / For K={num_kernels} / Minimum angular error={min_error:0.02f}° (est={best_est:0.02f} rads, gt={best_gt:0.02f} rads)')
mean_angular_error = computeMeanAngularError(np.array(estimates), baseline_angle)
print(f'**Method Bayes-RPE ** / For K={num_kernels} / Mean angular error={min_error:0.02f}° (est={best_est:0.02f} rads, gt={best_gt:0.02f} rads)')
mean_angular_error = computeMeanAngularError(np.array(estimates), baseline_angle, evaluate_over_all_gt_angles=False)
print(f'**Method Bayes-RPE ** / For K={num_kernels} / Mean angular error={min_error:0.02f}° (est={best_est:0.02f} rads, gt={best_gt:0.02f} rads)')